In [5]:
import warnings
import os
warnings.filterwarnings('ignore')

In [6]:
from finetuning.config import Config
from finetuning.dataset_builder import DatasetBuilder
from finetuning.inference import InferenceEngine
from finetuning.model_manager import ModelManager
from finetuning.pdf_processor import PDFExtractor, TextCleaner

In [7]:
from pprint import pprint
config = Config()
config.initialize()

pprint(config)

Config(pdf_path='content/Metformin-Lipid-Therapy-Knowledge.pdf',
       model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T',
       output_dir='content\\pharma_tinyllama_lora_output',
       adapter_dir='content\\pharma_tinyllama_lora_adapter',
       processed_data_dir='content\\pharma_processed_data',
       min_chars_per_paragraph=80,
       block_size=512,
       test_size=0.15,
       seed=42,
       lora_r=16,
       lora_alpha=32,
       lora_dropout=0.05,
       num_train_epochs=10.0,
       per_device_train_batch_size=1,
       per_device_eval_batch_size=1,
       gradient_accumulation_steps=8,
       learning_rate=0.0002,
       warmup_ratio=0.03,
       weight_decay=0.01,
       logging_steps=1,
       logging_first_step=True,
       eval_steps=10,
       save_steps=25,
       save_total_limit=2,
       max_steps=-1)


In [8]:
extractor = PDFExtractor(config.pdf_path)
pages = extractor.extract_pages()
print(f"Extracted {len(pages)} pages from PDF.")
pprint(pages[0])

Extracted 6 pages from PDF.
{'char_count': 2244,
 'page': 1,
 'text': 'Metformin is one of the most widely prescribed oral '
         'antihyperglycemic agents.\u200b\n'
         ' Its primary mechanism of action involves the activation of '
         'AMP-activated protein kinase \n'
         '(AMPK), a central metabolic regulator that promotes glucose uptake '
         'and fatty acid oxidation \n'
         'while inhibiting hepatic gluconeogenesis.\u200b\n'
         ' Beyond its glycemic control, Metformin has been shown to improve '
         'cardiovascular outcomes \n'
         'and display anti-inflammatory properties.\u200b\n'
         ' Recent studies also suggest potential anticancer effects through '
         'inhibition of the mTOR \n'
         'signaling pathway and suppression of tumor angiogenesis. \n'
         ' \n'
         'Clinical trials have demonstrated that combining Atorvastatin with '
         'Ezetimibe results in \n'
         'significant reductions in low-dens

In [9]:
cleaner = TextCleaner(min_chars_per_paragraph=config.min_chars_per_paragraph)
cleaned_pages = cleaner.clean_pages(pages)
paragraph_records = cleaner.split_into_paragraph_records(cleaned_pages)
print(f"Created {len(paragraph_records)} paragraph records.")
pprint(paragraph_records[0])

Created 9 paragraph records.
{'char_count': 575,
 'paragraph_id': 1,
 'source_page': 1,
 'text': 'Metformin is one of the most widely prescribed oral '
         'antihyperglycemic agents. Its primary mechanism of action involves '
         'the activation of AMP-activated protein kinase (AMPK), a central '
         'metabolic regulator that promotes glucose uptake and fatty acid '
         'oxidation while inhibiting hepatic gluconeogenesis. Beyond its '
         'glycemic control, Metformin has been shown to improve cardiovascular '
         'outcomes and display anti-inflammatory properties. Recent studies '
         'also suggest potential anticancer effects through inhibition of the '
         'mTOR signaling pathway and suppression of tumor angiogenesis.'}


In [10]:
cleaner.save_jsonl(cleaned_pages, os.path.join(config.processed_data_dir, "pdf_pages_cleaned.jsonl"))
cleaner.save_jsonl(paragraph_records, os.path.join(config.processed_data_dir, "paragraph_records.jsonl"))

In [11]:
model_manager = ModelManager(config)
tokenizer = model_manager.load_tokenizer()
builder = DatasetBuilder(config, tokenizer)

dataset = builder.build_dataset(paragraph_records)
tokenized = builder.tokenize_dataset(dataset)
final_dataset = builder.create_training_blocks(tokenized)
final_dataset

Tokenizing text corpus: 100%|██████████| 2/2 [00:00<00:00, 646.97 examples/s]
Creating fixed-size training blocks of 512 tokens: 100%|██████████| 7/7 [00:00<00:00, 1399.10 examples/s]
Creating fixed-size training blocks of 512 tokens: 100%|██████████| 2/2 [00:00<00:00, 996.86 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 0
    })
})

In [12]:
data_collator = model_manager.build_data_collator(tokenizer)
base_model = model_manager.load_base_model()
peft_model = model_manager.create_peft_model(base_model)
trainer = model_manager.build_trainer(
    model=peft_model,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5799.81it/s]


In [13]:
print("Starting training. This may take a while.")
trainer.train()
model_manager.save_adapter(peft_model, tokenizer, config.adapter_dir)
print(f"Adapter saved to {config.adapter_dir}")

Starting training. This may take a while.


Step,Training Loss
1,2.095782
2,2.095782
3,2.095660
4,2.095418
5,2.095050
6,2.094560
7,2.093971
8,2.093217
9,2.092385
10,2.091453


Adapter saved to content\pharma_tinyllama_lora_adapter


In [14]:
inference_engine = InferenceEngine(config).load(config.adapter_dir)
sample_prompt = "Metformin is one of the most"
completion = inference_engine.generate_completion(sample_prompt, max_new_tokens=180)
print("Sample continuation:\n", completion)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7730.69it/s]
[transformers] Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample continuation:
 Metformin is one of the most common drugs used to treat diabetes.
FDA Approves First Treatment for Tourette Syndrome Tics: The Food and Drug Administration has approved a new treatment for people with tic disorders, such as Tourette syndrome (TS).
Renal Failure: What You Should Know Renal failure is when your kidneys stop working properly.
Anti-inflammatory Drugs May Help With Arthritis Treatment Arthritis can be debilitating and painful, and it can lead to other complications like joint deformities, osteoporosis and loss of mobility.
Learn About Medications For ADHD In children who are diagnosed with attention deficit hyperactivity disorder (ADHD), there are medications that can help them
